# 04 · Exploring GSE221390 (skin cancer atlas: the melanoma Visium samples)

Same task and structure as `01_explore_GSE316760.ipynb`. The big difference: **GEO has no images and no spot
coordinates for this dataset**, only count matrices. Sections 1–2 show how we recover the spot positions anyway.

**The study:** a multi-technology atlas of the three main skin cancers (squamous cell carcinoma SCC, basal cell
carcinoma BCC, melanoma) from the University of Queensland. On GEO there are 7 Visium samples (3 SCC, 2 BCC,
**2 melanoma**) and one single-cell RNA-seq dataset (SCC/BCC and healthy skin, no melanoma). We only use the
**2 melanoma Visium samples**; each holds sections from **two patients**.

### What is different from GSE316760

| | GSE316760 (notebook 01) | **GSE221390 (this notebook)** |
|---|---|---|
| Samples used | 2 | **2 Visium samples × 2 patients each** (66487 + 48974, 6474 + 21031) |
| Files on GEO | matrix + images + positions + scale factors | **matrix only** (`filtered_feature_bc_matrix.tar.gz`) |
| Images | H&E hires/lowres | **none** |
| Spot coordinates | `tissue_positions.csv` (pixels) | **none: recovered from the barcodes (section 2), in µm, without image** |
| Gene panel | 17,943 genes (FFPE probe set v1) | **the same 17,943-gene panel**, so these two datasets are directly comparable |
| Patients per sample | 1 | **2, mixed in one matrix; split by tissue piece (section 4), but which piece is which patient is unknown** |
| Other data in the series | — | SCC, BCC and scRNA-seq (not used here) |

In [ ]:
import sys
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)   # harmless deprecation notices from squidpy/anndata
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

# Project root = the spatial-reasoning folder (the one with pyproject.toml), wherever the notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "01_data_exploration"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.io import load_visium_without_images, read_series_matrix

GSE = "GSE221390"
RAW = ROOT / f"data/raw/{GSE}/files"
PROCESSED = ROOT / "data/processed"
FIGS = STAGE / f"results/{GSE}"
FIGS.mkdir(parents=True, exist_ok=True)
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)


def plot_grid(a, color, ax=None, title=None, **kw):
    """Spatial plot without an image: spots at their µm grid positions (y downwards, like an image)."""
    ax = sc.pl.embedding(a, basis="spatial", color=color, ax=ax, show=False, title=title, size=18, frameon=False, **kw)
    for axis in np.atleast_1d(ax):
        axis.invert_yaxis()
        axis.set_aspect("equal")
    return ax

## 1. What GEO gives us

Each Visium sample is a `.tar.gz` archive. Listing the melanoma ones:

In [ ]:
SAMPLES = {
    "Melanoma66487_48974": "GSM6863556_Melanoma66487_48974",
    "Melanoma6474_21031": "GSM6863557_Melanoma6474_21031",
}
for lib, prefix in SAMPLES.items():
    with tarfile.open(RAW / f"{prefix}_filtered_feature_bc_matrix.tar.gz") as tar:
        print(lib, "->", [m.name for m in tar.getmembers() if m.isfile()])

Only the three files of the count matrix: **no `tissue_positions.csv`, no scale factors, no images**. Without
positions this would be ordinary bulk-like data with no space, useless for our project.

## 2. Recovering the spot positions from the barcodes

Every Visium spot has a **barcode** (e.g. `AAACAAGTATCTCCCA-1`) and, for a given slide version, **each barcode
always sits at the same grid position** (`array_row`, `array_col`) on every slide. GSE316760 (notebook 01) used
the same slide version ("Visium V1 Slide") and its `tissue_positions.csv` lists all 4,992 barcodes with their grid
positions, so we can use it as a reference map.

Check first: are all barcodes of GSE221390 in that reference?

In [ ]:
REFERENCE = ROOT / "data/raw/GSE316760/files/GSM9459774_mel2_tissue_positions.csv.gz"
ref = pd.read_csv(REFERENCE, index_col=0)
print("reference grid:", len(ref), "barcodes")
for lib, prefix in SAMPLES.items():
    with tarfile.open(RAW / f"{prefix}_filtered_feature_bc_matrix.tar.gz") as tar:
        member = next(m for m in tar.getmembers() if m.name.endswith("barcodes.tsv.gz"))
        bc = pd.read_csv(tar.extractfile(member), header=None, compression="gzip")[0]
    print(f"{lib}: {bc.isin(ref.index).sum()} of {len(bc)} barcodes found in the reference grid")

**All barcodes match**, so the positions are recoverable. `load_visium_without_images` (in `src/space_mel/io.py`)
does this and converts grid positions to micrometres with the physical Visium geometry (`space_mel.spatial.grid_to_um`):
one column step = 50 µm, one row step = 86.6 µm, so neighbouring spots are 100 µm apart.

**Caveat:** without the image we do not know how the slide was oriented under the microscope; the layout may be
mirrored or rotated relative to the real tissue. **Distances are exact**; directions ("left", "top") are not
meaningful.

In [ ]:
adatas = {
    lib: load_visium_without_images(RAW / f"{prefix}_filtered_feature_bc_matrix.tar.gz", REFERENCE,
                                    PROCESSED / f"visium/{GSE}_{lib}", library_id=lib)
    for lib, prefix in SAMPLES.items()
}
adatas["Melanoma66487_48974"]

## 3. What is inside, and metadata

Same slots as before, except `uns["spatial"]` has **no images and no scale factors**, and `obsm["spatial"]` is
already in **µm** (not pixels).

In [ ]:
for lib, a in adatas.items():
    X = a.X
    is_int = np.allclose(X.data[:1000], np.round(X.data[:1000]))
    print(f"{lib}: {a.n_obs} spots x {a.n_vars} genes | {X.nnz / (a.n_obs * a.n_vars):.1%} non-zero | "
          f"integer counts: {is_int} | MT- genes: {a.var_names.str.startswith('MT-').sum()}")
display(adatas["Melanoma66487_48974"].obs.head())

meta = read_series_matrix(ROOT / f"data/raw/{GSE}/metadata/{GSE}_series_matrix.txt.gz")
with pd.option_context("display.max_colwidth", 300):
    display(meta.loc[meta.index.str.startswith("Melanoma"), ["geo_accession", "characteristics_ch1", "characteristics_ch1_2"]])
    print(meta["extract_protocol_ch1"].iloc[0][:1200])

**Reading this:**
- ~4,000 spots and **17,943 genes**: exactly the FFPE probe panel of GSE316760, no `MT-` genes (so no %MT QC).
- The melanoma blocks are FFPE, collected 2008–2018 (from the extraction protocol text above).
- GEO metadata is generic ("skin", "multiple cell types"): no clinical annotation, no region labels.
- **Another GEO inconsistency:** the sample title says patient "6474", the protocol text says "6747".

## 4. Separating the two patients

Each sample contains two patients' sections. On the grid they should appear as **two separate pieces of tissue**.
We find the pieces by linking spots that touch (centres ≤ 100 µm apart) and taking the connected groups.

In [ ]:
for lib, a in adatas.items():
    xy = a.obsm["spatial"]
    touching = cKDTree(xy).sparse_distance_matrix(cKDTree(xy), 101) > 0
    n, labels = connected_components(touching, directed=False)
    sizes = pd.Series(labels).value_counts()
    names = {old: f"piece {i + 1}" for i, old in enumerate(sizes.index)}   # piece 1 = largest
    a.obs["piece"] = pd.Categorical([names[l] for l in labels])
    print(f"{lib}: {n} pieces, sizes {a.obs.piece.value_counts().to_dict()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (lib, a) in zip(axes, adatas.items()):
    plot_grid(a, "piece", ax=ax, title=f"{lib}: tissue pieces")
fig.savefig(FIGS / "tissue_pieces.png", bbox_inches="tight")

Each sample splits cleanly into **exactly two pieces**, one per patient. GEO does not tell us which piece belongs
to which patient ID (e.g. 66487 vs 48974), so we call them *piece 1* (larger) and *piece 2*. For analyses we
should treat each piece as a separate patient.

## 5. Coordinate units

Here there is nothing to check against an image: the coordinates come straight from the physical grid, so they
are in µm by construction. We copy them to `obsm["spatial_um"]` for consistency with the other notebooks and
verify the spacing.

In [ ]:
rows = {}
for lib, a in adatas.items():
    a.obsm["spatial_um"] = a.obsm["spatial"].copy()
    xy = a.obsm["spatial_um"]
    nn = np.median(cKDTree(xy).query(xy, k=2)[0][:, 1])
    rows[lib] = {"median neighbour distance (um)": nn,
                 "extent x * y (mm)": "{:.2f} x {:.2f}".format(*np.ptp(xy, axis=0) / 1000)}
    a.uns["spatial_units"] = {"um_per_px": None, "method": "Visium grid from barcodes (no image)"}
pd.DataFrame(rows).T

## 6. Quality control

In [ ]:
for lib, a in adatas.items():
    sc.pp.calculate_qc_metrics(a, inplace=True, percent_top=None)

qc = pd.concat({lib: a.obs[["piece", "total_counts", "n_genes_by_counts"]] for lib, a in adatas.items()},
               names=["sample"]).reset_index(level=0)
display(qc.groupby(["sample", "piece"], observed=True)[["total_counts", "n_genes_by_counts"]].median().round(0))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for lib in adatas:
    axes[0].hist(qc.loc[qc["sample"] == lib, "total_counts"], bins=80, alpha=0.6, label=lib)
    axes[1].hist(qc.loc[qc["sample"] == lib, "n_genes_by_counts"], bins=80, alpha=0.6, label=lib)
axes[0].set_xlabel("total counts per spot"); axes[1].set_xlabel("genes detected per spot")
axes[0].legend()
fig.savefig(FIGS / "qc_histograms.png", bbox_inches="tight")

In [ ]:
for lib, a in adatas.items():
    plot_grid(a, ["total_counts", "n_genes_by_counts"], title=[f"{lib}: total_counts", f"{lib}: n_genes_by_counts"])
    plt.savefig(FIGS / f"qc_spatial_{lib}.png", bbox_inches="tight")
    plt.show()

## 7. Marker genes

Same panel as notebooks 02–03 (all genes are in this panel).

In [ ]:
MARKERS = ["MLANA", "PMEL", "SOX10", "PRAME", "MKI67", "PTPRC", "CD3E", "CD8A", "CD68", "CD163", "CD74",
           "COL1A1", "KRT14"]

for lib, a in adatas.items():
    a.layers["counts"] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    plot_grid(a, MARKERS, ncols=7)
    plt.savefig(FIGS / f"markers_{lib}.png", bbox_inches="tight")
    plt.show()

## 8. A first data-driven segmentation (clustering)

Same procedure as the other notebooks, per sample.

In [ ]:
for lib, a in adatas.items():
    sc.pp.highly_variable_genes(a, n_top_genes=2000, flavor="seurat_v3", layer="counts")
    sc.pp.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a)
    sc.tl.leiden(a, resolution=0.5, flavor="igraph", n_iterations=2, key_added="leiden")
    sc.tl.umap(a)
    print(lib, a.obs.leiden.value_counts().sort_index().to_dict())

In [ ]:
for lib, a in adatas.items():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sc.pl.umap(a, color="leiden", ax=axes[0], show=False, title=f"{lib}: UMAP")
    plot_grid(a, "leiden", ax=axes[1], title=f"{lib}: clusters in space")
    fig.savefig(FIGS / f"clusters_{lib}.png", bbox_inches="tight")

In [ ]:
for lib, a in adatas.items():
    sc.pl.matrixplot(a, MARKERS, groupby="leiden", standard_scale="var", title=lib, cmap="viridis", show=False)
    plt.savefig(FIGS / f"cluster_markers_{lib}.png", bbox_inches="tight")
    plt.show()
    sc.tl.rank_genes_groups(a, "leiden", method="wilcoxon")
    print(f"--- {lib}: top 8 genes per cluster")
    display(pd.DataFrame(a.uns["rank_genes_groups"]["names"]).head(8))

### Are the recovered positions right?

The clusters were computed from expression only, without positions. If the barcode → position mapping were wrong,
spots would land in random places and neighbouring spots would rarely share a cluster. If it is right, real tissue
structures (epidermis, glands, tumour) make **neighbours much more similar than chance**. We compare the fraction of
touching spot pairs in the same cluster with the same number after shuffling the positions 100 times.

In [ ]:
rng = np.random.default_rng(0)
rows = {}
for lib, a in adatas.items():
    pairs = cKDTree(a.obsm["spatial"]).query_pairs(101, output_type="ndarray")   # touching spots
    labels = a.obs["leiden"].to_numpy()
    observed = np.mean(labels[pairs[:, 0]] == labels[pairs[:, 1]])
    shuffled = [np.mean((s := rng.permutation(labels))[pairs[:, 0]] == s[pairs[:, 1]]) for _ in range(100)]
    rows[lib] = {"neighbour pairs": len(pairs), "same cluster (observed)": observed,
                 "same cluster (shuffled, mean)": np.mean(shuffled), "same cluster (shuffled, max)": np.max(shuffled)}
pd.DataFrame(rows).T.round(3)

If the observed value is far above every shuffled value, the clusters form coherent regions on the recovered grid,
which can only happen if the positions are correct (or at least preserve real neighbourhoods).

## 9. Save

One compressed `.h5ad` per sample (both pieces together, `obs["piece"]` says which is which).

In [ ]:
for lib, a in adatas.items():
    path = PROCESSED / f"{GSE}_{lib}.h5ad"
    a.write_h5ad(path, compression="gzip")
    print("saved", path.relative_to(ROOT), f"({path.stat().st_size / 1e6:.0f} MB)")

## 10. Summary and open questions

**What the data contains (task checklist)**
- **Expression matrices:** 2 Visium samples × 2 patients each (4,093 and 3,901 spots) × 17,943 genes (the same FFPE
  probe panel as GSE316760, no `MT-` genes). Shallow: median 1,259–4,347 counts per spot depending on the piece.
- **Metadata:** FFPE blocks from 2008–2018; generic GEO annotation. GEO spells one patient "6474" and "6747".
- **Spatial coordinates:** **not on GEO**. Recovered from the barcodes with GSE316760's grid as reference (100% of
  barcodes match), directly in µm (neighbours exactly 100 µm apart). **Orientation unknown** (may be mirrored or
  rotated); distances are reliable.
- **Validation:** clusters computed from expression alone put touching spots in the same cluster 79–82% of the time,
  vs 26–27% after shuffling positions, so the recovered positions preserve real tissue structure.
- **Tissue images:** none.
- **Annotations:** none; each sample splits into two tissue pieces (one per patient), but which is which is unknown.

**First biological look** (hypotheses)
- Mostly **normal skin structures**: epidermis, dermis, sweat and sebaceous glands, hair follicles, vessels.
- Melanocytic clusters are small: 148 spots (Melanoma66487_48974, cluster 7) and 230 spots (Melanoma6474_21031,
  cluster 5). In Melanoma66487_48974 a band along one border of piece 1 combines melanocytic genes with immune
  genes (`PTPRC`, `CD3E`, `CD74`).

**Open questions / next steps**
1. Ask whether images / positions exist elsewhere (paper supplement, data portal).
2. Stage and type of these melanomas (paper).
3. Limited value for tumour–immune distance questions; possibly useful as a "little or no tumour" control.